In [216]:
import json 
import os
import pandas as pd
import chardet
from sklearn.metrics.pairwise import cosine_similarity
import itertools
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [229]:
data_path = "/Users/carolgao/MIT Dropbox/Carol Gao/common_experience_2025/"

In [230]:
with open(f"{data_path}/contracts/South Shore Imported Cars, Inc v. Volkswagen of America, Inc.txt", "rb") as f:
    raw_data = f.read()
    result = chardet.detect(raw_data)
    encoding = result["encoding"]

In [231]:
CONTRACT = "2277CV00212"

In [232]:
with open(f"{data_path}/contracts/corrected_output_{CONTRACT}.txt", "r", encoding='ISO-8859-1') as f:
    text_str = f.read()

In [233]:
def split_paragraphs_merge_short(text, min_length=1000):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip() != ""]
    chunks = []
    buffer = ""
    
    for para in paragraphs:
        if len(buffer) > 0:
            candidate = buffer + "\n\n" + para
        else:
            candidate = para
        
        if len(candidate) < min_length:
            buffer = candidate
        else:
            chunks.append(candidate)
            buffer = ""
    
    # Add any remaining buffer
    if buffer:
        chunks.append(buffer)
    
    return chunks

### Compute similarity

In [234]:
df_contract = pd.read_csv(f"{data_path}/contracts/{CONTRACT}_embeddings.csv")
df_ruling = pd.read_csv(f"{data_path}/rulings/case_embeddings.csv")

In [235]:
df_contract_matrix = df_contract.drop(columns = {"chunk"})
df_ruling_matrix = df_ruling.drop(columns = {"case", "flaw_index"})

In [236]:
pairs = list(itertools.product(df_contract_matrix.index, df_ruling_matrix.index))

In [237]:
cos_sim_matrix = pd.DataFrame(cosine_similarity(df_contract_matrix, df_ruling_matrix).flatten()
                             , index=pd.MultiIndex.from_tuples(pairs, names=['contract_index', 'ruling_index'])).rename(columns = {0: "cos_sim"})

In [238]:
# retrive top 5 rulings 
top_indices = cos_sim_matrix.sort_values(by = 'cos_sim', ascending = False).head(5).index

In [239]:
top_cases = pd.DataFrame(columns = ["contract_chunk", "case", "case_flaw_index"])
for i in range(0, len(top_indices)):
    contract_chunk = df_contract.loc[df_contract.index == top_indices[i][0]]['chunk']
    case = df_ruling.loc[df_ruling.index == top_indices[i][1]]['case']
    case_flaw = df_ruling.loc[df_ruling.index == top_indices[i][1]]['flaw_index']
    top_cases = pd.concat([top_cases, pd.DataFrame([np.array([contract_chunk, case, case_flaw]).flatten()], columns = ["contract_chunk", "case", "case_flaw_index"])])

In [240]:
chunks = split_paragraphs_merge_short(text_str)

In [241]:
top_cases['contract'] = ""
for i in top_cases['contract_chunk'].unique():
    top_cases.loc[top_cases['contract_chunk'] == i, 'contract'] = chunks[i]

In [253]:
top_cases

,contract_chunk,case,case_flaw_index,contract
0,9,"Larson v. Landvest, Inc., 19 Mass. L. Rep. 479...",2,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...
0,13,"Moore v. Galante, 14 Mass. L. Rep. 264.json",1,SELLER represents to the best of their knowled...
0,9,"Driscoll v. Toll MA Ltd. P'ship, 26 Mass. L. R...",0,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...
0,9,"SW-NEC UP LENDER LLC v. Rockland Meadows, LLC,...",1,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...
0,13,"Hernandez v. Lesburg, 20 LCR 207.json",0,SELLER represents to the best of their knowled...


In [ ]:
top_cases.to_csv(f"{data_path}/contracts/top_pairs_{CONTRACT}_new.csv", index = False)

In [275]:
def normalize_rows(X, eps=1e-12):
    """
    Ensure rows are nonnegative distributions.
    Adds smoothing to avoid zeros and negatives.
    """
    X = np.clip(X, a_min=0, a_max=None)   # remove negatives
    X = X + eps                          # avoid zeros
    return X / X.sum(axis=1, keepdims=True)

In [278]:
from scipy.special import rel_entr 
def pairwise_kl(df1, df2, eps=1e-12):
    """
    Compute pairwise symmetric KL divergence between rows of df1 and df2.
    Returns a DataFrame of shape (len(df1), len(df2)).
    """
    P = normalize_rows(df1.to_numpy(dtype=float))
    Q = normalize_rows(df2.to_numpy(dtype=float))
    # add smoothing and normalize
    P = (P + eps) / (P + eps).sum(axis=1, keepdims=True)
    Q = (Q + eps) / (Q + eps).sum(axis=1, keepdims=True)

    m, d = P.shape
    n = Q.shape[0]

    sym_kl_matrix = np.zeros((m, n))

    for i in range(m):
        for j in range(n):
            kl_pq = np.sum(rel_entr(P[i], Q[j]))  # D(P || Q)
            kl_qp = np.sum(rel_entr(Q[j], P[i]))  # D(Q || P)
            sym_kl_matrix[i, j] = kl_pq + kl_qp

    return pd.DataFrame(sym_kl_matrix, index=df1.index, columns=df2.index)

In [284]:

def top_k_similar(df1, df2, k=5):
    """Find k most similar pairs based on symmetric KL divergence."""
    kl_df = pairwise_kl(df1, df2)

    # flatten to list of (row_df1, row_df2, divergence)
    pairs = [
        (i, j, kl_df.loc[i, j]) 
        for i in kl_df.index 
        for j in kl_df.columns
    ]

    # sort by divergence (lower = more similar)
    pairs_sorted = sorted(pairs, key=lambda x: x[2])

    return pairs_sorted[:k]

In [286]:
top_5_KL = top_k_similar(df_contract_matrix, df_ruling_matrix, k=5)

In [288]:
df_top_5_KL = pd.DataFrame()
for i in range(0,5): 
    contract_chunk = df_contract.loc[df_contract.index == top_5_KL[i][0]]['chunk']
    case = df_ruling.loc[df_ruling.index == top_5_KL[i][1]]['case']
    case_flaw = df_ruling.loc[df_ruling.index == top_5_KL[i][1]]['flaw_index']
    top_cases = pd.concat([top_cases, pd.DataFrame([np.array([contract_chunk, case, case_flaw]).flatten()], columns = ["contract_chunk", "case", "case_flaw_index"])])

13

In [293]:
df_contract["chunk"][contract_index]

SyntaxError: unterminated string literal (detected at line 1) (1163138791.py, line 1)

In [249]:
df = pd.read_csv(f"{data_path}/contracts/top_pairs_{CONTRACT}_new.csv")
reasoning = pd.read_csv(f"{data_path}/contracts/reasoning_{CONTRACT}_new.csv")
df['case_summary'] = ""
for i in range(0, len(df)):
    contract_text = df['contract'][i]
    case_name = df['case'][i]
    case_flaw_index = df['case_flaw_index'][i]
    with open(f"{data_path}/rulings/Trial_678_rulings_output/{case_name}", "r") as f:
        data = json.load(f)
    case_text = str(data['flaws'][case_flaw_index])
    df['case_summary'][i] = case_text
final_df = pd.concat([df, reasoning],axis =1)
final_df = final_df.rename(columns = {"0": "reasoning"})

In [251]:
df = pd.read_csv(f"{data_path}/contracts/top_pairs_{CONTRACT}.csv")
reasoning = pd.read_csv(f"{data_path}/contracts/reasoning_{CONTRACT}.csv")
df['case_summary'] = ""
for i in range(0, len(df)):
    contract_text = df['contract'][i]
    case_name = df['case'][i]
    case_flaw_index = df['case_flaw_index'][i]
    with open(f"{data_path}/rulings/Trial_678_rulings_output/{case_name}", "r") as f:
        data = json.load(f)
    case_text = str(data['flaws'][case_flaw_index])
    df['case_summary'][i] = case_text
final_df = pd.concat([df, reasoning],axis =1)
final_df = final_df.rename(columns = {"0": "reasoning"})
final_df

,contract_chunk,case,case_flaw_index,contract,case_summary,reasoning
0,9,"SW-NEC UP LENDER LLC v. Rockland Meadows, LLC,...",2,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...,"{'type': 'Marketing fee assignment ambiguity',...",The contract section under review includes war...
1,4,"Feeney v. Dell, 454 Mass. 192.json",0,Page3\n\nSEL2'Initials\n\nPURCHASE AND SALE AG...,{'type': 'Arbitration clause prohibiting class...,"The section from the ""Purchase and Sale Agreem..."
2,13,"Marmik, LLC v. Packer, 104 Mass. App. Ct. 1117...",0,SELLER represents to the best of their knowled...,"{'type': 'Ambiguity in scope of guaranty', 'or...",The section of the contract outlines the Selle...
3,9,"Dapkus v. Dapkus, 2008 Mass. App. Unpub. LEXIS...",1,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...,{'type': 'Discretionary distribution on trust ...,The contract section outlines warranties and r...
4,4,"Meredith & Grew, Inc. v. Worcester Lincoln, LL...",1,Page3\n\nSEL2'Initials\n\nPURCHASE AND SALE AG...,{'type': 'Ambiguous scope of brokerage service...,The contract section outlines the conditions u...


In [250]:
final_df

,contract_chunk,case,case_flaw_index,contract,case_summary,reasoning
0,9,"Larson v. Landvest, Inc., 19 Mass. L. Rep. 479...",2,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...,"{'type': 'As-is and no-warranties clause', 'or...",This section of the Purchase and Sale Agreemen...
1,13,"Moore v. Galante, 14 Mass. L. Rep. 264.json",1,SELLER represents to the best of their knowled...,{'type': 'Lack of definite identification of l...,The contract section indicates that the SELLER...
2,9,"Driscoll v. Toll MA Ltd. P'ship, 26 Mass. L. R...",0,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...,{'type': 'Exclusive remedy and limitation of l...,This contract section outlines the warranties ...
3,9,"SW-NEC UP LENDER LLC v. Rockland Meadows, LLC,...",1,CE]\n11Â·07 AM EST\nBl.JYEltopsâ¢ffi.hials\tS...,"{'type': 'Lis pendens waiver clause', 'origina...",The provided contract section outlines the war...
4,13,"Hernandez v. Lesburg, 20 LCR 207.json",0,SELLER represents to the best of their knowled...,{'type': 'Ambiguous severance procedure for pa...,The section of the contract outlines the repre...


In [248]:
final_df[['contract', 'case_summary', 'reasoning']].to_csv(f"{data_path}/contracts/final_reasoning_{CONTRACT}_new.csv", index = False)